# Project 4 — Local Resume Rewriter

## Improve Resume Bullets and Tailor Wording with a Local LLM

**Goal:** Take rough resume bullet points, critique them, rewrite them using
the STAR method, and tailor them to a target job description — all locally.

**Stack:** Ollama · LangChain · Jupyter

### What You'll Learn

1. **Critique** resume bullets for impact, clarity, and specificity
2. **Rewrite** bullets using the STAR (Situation, Task, Action, Result) method
3. **Tailor** bullets to match a target job description's keywords
4. **Score** resume-JD alignment quantitatively
5. Compare **different prompting strategies** for rewriting

### Prerequisites

- Ollama running with `qwen3:8b` pulled

In [ ]:
# Install dependencies (uncomment and run once)
# !pip install -q langchain langchain-ollama langchain-core

## Step 1 — Verify Ollama and Configure LLM

In [ ]:
import requests
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print(f"Ollama running — {len(r.json().get('models', []))} models")
except: print("Start Ollama first!")

llm = ChatOllama(model="qwen3:8b", temperature=0.3)
print("LLM configured")

## Step 2 — Sample Resume Bullets

These are typical **weak** resume bullets that many people write.
They lack specificity, measurable impact, and action verbs.

In [ ]:
weak_bullets = [
    "Responsible for managing a team and delivering projects on time.",
    "Worked on data analysis and made reports for management.",
    "Helped improve the company website and increased traffic.",
    "Used Python and SQL for various data tasks.",
    "Participated in code reviews and helped maintain code quality.",
]

print("=== Original Resume Bullets ===")
for i, b in enumerate(weak_bullets, 1):
    print(f"  {i}. {b}")

## Step 3 — Critique the Bullets

Before rewriting, let's have the LLM identify what's weak about
each bullet. This teaches the model (and us) what to fix.

In [ ]:
critique_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a resume coach. For each bullet point, provide a brief critique:
- Is it specific or vague?
- Does it show measurable impact?
- Does it use strong action verbs?
- What is missing?

Be concise — 2-3 sentences per bullet."""),
    ("human", "Critique these resume bullets:\n\n{bullets}"),
])

bullets_text = "\n".join(f"{i}. {b}" for i, b in enumerate(weak_bullets, 1))
critique = (critique_prompt | llm | StrOutputParser()).invoke({"bullets": bullets_text})

print("=== Critique ===")
print(critique)

## Step 4 — Rewrite Using STAR Method

The **STAR method** (Situation, Task, Action, Result) creates impactful
bullet points. We ask the LLM to transform each weak bullet into a
strong STAR-formatted version.

In [ ]:
star_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a resume writing expert. Rewrite each bullet point using the STAR method:
- **Situation**: Brief context
- **Action**: What you specifically did (strong verb)
- **Result**: Measurable outcome with numbers when possible

Make reasonable assumptions about metrics. Keep each bullet to 1-2 lines.
Return ONLY the rewritten bullets, numbered."""),
    ("human", "Rewrite these resume bullets:\n\n{bullets}"),
])

rewritten = (star_prompt | llm | StrOutputParser()).invoke({"bullets": bullets_text})

print("=== STAR-Rewritten Bullets ===")
print(rewritten)

## Step 5 — Tailor to a Job Description

Now let's align the rewritten bullets with a specific job posting.
The model should emphasize **keywords and skills** from the JD.

In [ ]:
sample_jd = """
Senior Data Engineer — TechCorp

Requirements:
- 5+ years experience with Python, SQL, and cloud data platforms
- Experience building and maintaining ETL/ELT pipelines
- Strong understanding of data modeling and warehousing
- Experience with Apache Spark, Airflow, or similar tools
- Track record of improving data quality and pipeline reliability
- Leadership experience mentoring junior engineers
- Excellent communication skills for cross-functional collaboration
"""

tailor_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a resume tailoring expert. Given a job description and resume bullets,
rewrite the bullets to better match the job requirements. Emphasize relevant
keywords, skills, and experiences from the JD. Keep the content truthful
but strategically aligned."""),
    ("human", "Job Description:\n{jd}\n\nResume Bullets to tailor:\n{bullets}"),
])

tailored = (tailor_prompt | llm | StrOutputParser()).invoke({
    "jd": sample_jd,
    "bullets": rewritten,
})

print("=== Tailored for Data Engineer Role ===")
print(tailored)

## Step 6 — Score Resume-JD Fit

Let's ask the model to evaluate how well the tailored resume
matches the job description, with a numerical score.

In [ ]:
score_prompt = ChatPromptTemplate.from_messages([
    ("system", """Rate how well these resume bullets match the job description.

Score each criterion from 1-10:
1. Keyword alignment (do the bullets mention JD skills?)
2. Experience level match
3. Impact/metrics quality
4. Overall fit

Provide scores and a brief explanation for each."""),
    ("human", "Job Description:\n{jd}\n\nResume Bullets:\n{bullets}"),
])

# Score original vs tailored
print("=== Scoring Original Bullets ===")
orig_score = (score_prompt | llm | StrOutputParser()).invoke({"jd": sample_jd, "bullets": bullets_text})
print(orig_score)

print("\n=== Scoring Tailored Bullets ===")
tailored_score = (score_prompt | llm | StrOutputParser()).invoke({"jd": sample_jd, "bullets": tailored})
print(tailored_score)

## Step 7 — Side-by-Side Comparison

In [ ]:
print("=== Before vs After ===\n")
print("ORIGINAL:")
for i, b in enumerate(weak_bullets, 1):
    print(f"  {i}. {b}")
print("\nTAILORED:")
print(tailored)

## Limitations & Tradeoffs

| Aspect | What happens | How to improve |
|--------|-------------|----------------|
| **Fabricated metrics** | Model may invent numbers | Ask user to provide real metrics |
| **Keyword stuffing** | Over-optimizing for JD keywords | Balance relevance with authenticity |
| **Generic rewrites** | Without context, rewrites may be vague | Provide role-specific context |
| **Thinking tags** | qwen3 may include reasoning in output | Post-process or use non-thinking model |

### What this project does NOT cover
- ATS (Applicant Tracking System) parsing simulation
- Multi-page resume formatting
- Portfolio/project section optimization

## What You Learned

1. **Resume critique** — identifying weaknesses before rewriting
2. **STAR method** — structured approach to impactful bullet points
3. **JD tailoring** — aligning resume language with job requirements
4. **Fit scoring** — quantitative evaluation of resume-JD alignment
5. **Before/after comparison** — measuring improvement visually

## Exercises

1. **Use your own resume** — paste your actual bullets and run the pipeline
2. **Try multiple JDs** — tailor the same resume for 3 different roles
3. **Add a skills section** — extend the pipeline to also rewrite a skills summary
4. **Iterative refinement** — feed the critique back and rewrite again

---

*Next project: **05 — Local Cover Letter Generator** (multi-input context synthesis)*